In [6]:
pip install torch

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install torchvision

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install transformers


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install torch torchvision transformers opencv-python


   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   --- ------------------------------------ 3.1/39.0 MB 18.5 MB/s eta 0:00:02
   -------- ------------------------------- 7.9/39.0 MB 20.3 MB/s eta 0:00:02
   ----------- ---------------------------- 11.5/39.0 MB 19.0 MB/s eta 0:00:02
   ---------------- ----------------------- 16.5/39.0 MB 20.8 MB/s eta 0:00:02
   ---------------------- ----------------- 21.5/39.0 MB 20.9 MB/s eta 0:00:01
   ------------------------- -------------- 24.4/39.0 MB 19.5 MB/s eta 0:00:01
   ---------------------------- ----------- 28.0/39.0 MB 19.6 MB/s eta 0:00:01
   -------------------------------- ------- 31.5/39.0 MB 19.2 MB/s eta 0:00:01
   ------------------------------------- -- 36.2/39.0 MB 19.3 MB/s eta 0:00:01
   ---------------------------------------  38.8/39.0 MB 19.6 MB/s eta 0:00:01
   ---------------------------------------- 39.0/39.0 MB 17.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# ==========================
# Real-Time Webcam Emotion Detection (PyTorch)
# ==========================

import torch
import torch.nn as nn
from torchvision import transforms
from transformers import AutoModel, AutoConfig
import cv2

# ==========================
# 1. Device Setup
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================
# 2. Load Base Model Architecture
# ==========================
model_name = "HardlyHumans/Facial-expression-detection"
config = AutoConfig.from_pretrained(model_name)
base_model = AutoModel.from_config(config)

# ==========================
# 3. Define Emotion Model (5 classes assumed here)
# ==========================
num_classes = 5
emotion_labels = ["Angry", "Happy", "Neutral", "Sad", "Surprise"]  # Update as needed

class EmotionModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super(EmotionModel, self).__init__()
        self.base = base_model
        hidden_size = base_model.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        outputs = self.base(x)
        pooled = outputs.last_hidden_state[:, 0]
        return self.classifier(pooled)

# Instantiate and load weights
model = EmotionModel(base_model, num_classes).to(device)
model.load_state_dict(torch.load("finetuned_real.pth", map_location=device))
model.eval()
print("✅ Loaded finetuned_real.pth successfully.")

# ==========================
# 4. Preprocessing transforms
# ==========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# ==========================
# 5. Real-Time Webcam Detection
# ==========================
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = transforms.ToPILImage()(img_rgb)
    img_tensor = transform(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        _, pred = torch.max(outputs, 1)
        label = emotion_labels[pred.item()]

    cv2.putText(frame, f"Emotion: {label}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Real-Time Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Using device: cpu
✅ Loaded finetuned_real.pth successfully.
Press 'q' to quit.
